<a href="https://colab.research.google.com/github/Samarjamal326/Flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

**Lane:** CTR / Engagement Opportunity Scoring  
**Model audited:** Week-5 Logistic Regression  
**Decision window:** February 2026  
**Outcome window:** March 2026

This notebook audits the Week-5 model rather than introducing a new modeling idea. It does four things:

1. Reviews two findings from the FlyRank research paper and asks constructive methodology questions.
2. Re-runs the Week-5 model with a naive row split (**before**) and a client-grouped split (**after**).
3. Audits features for target/future leakage and checks the grouped split mechanically.
4. Reviews real failures and rewrites claims so they stay at the level of **observed / measured / directional / decision-support** evidence.

The goal is not to make the model look better. The goal is to make the evaluation harder to fool.


## 1. Two paper findings + my methodology questions

### Finding A — Growth Prediction

The FlyRank paper reports that a model trained on **96.6K clearly growing or declining pages** was about **90% accurate on unseen pages from the same brands** and **75% on brands not seen before**. The paper also says the test was repeated across 20 approaches and reports a 24.1K test-page figure. citeturn5view0turn5view2

**Methodology question:** I would want the exact construction of the “growing” and “declining” label: the comparison windows, threshold, exclusions, and the point at which the outcome becomes known. If the label is derived from a future traffic window, every feature must be demonstrably available before that window. I would also want the brand-held-out procedure to be the primary generalization claim when the intended use includes unseen brands.

This is a constructive reproducibility question, not a criticism of the result.

### Finding B — CTR by Position Tier

The paper reports weighted CTR of **0.420% for Top 3**, **0.340% for Page 1 (4–10)**, **0.325% for Striking Distance**, **0.163% for Page 3–5**, and **0.050% for Deep**. It labels the position-tier comparison confirmed and later reports a Kruskal-Wallis test for Position Tier → CTR. citeturn4view0turn5view1

**Methodology question:** because CTR and position are measured from the same search observations, I would ask whether the claim is intended as a descriptive association or as evidence that moving a page to a better position *causes* the click lift. For an ML label, I would also want the exact outcome window and whether the position-tier benchmark is estimated only from information available before that outcome window.

Again, the respectful distinction is between an observed relationship and a causal claim. The paper itself explicitly describes the study as a pattern study rather than proof of cause and effect. citeturn4view0


## 2. My model under an honest split — before / after

### Audit design

**Before:** random 80/20 row split. This is convenient, but pages from the same client can appear in both train and test.

**After:** 80/20 `GroupShuffleSplit` by `client_hash_id`. No client is allowed in both train and test.

To make the comparison focus on validation design, both experiments use the **same frozen February position-tier benchmark**, estimated only from the training clients of the honest grouped split. The March outcome label is then fixed once and reused for both split designs.

The target is:

> `1` if March future CTR is below the February position-tier benchmark; otherwise `0`.

This is a proxy prioritization outcome, not a causal refresh outcome.


In [1]:
# Setup: clone repo, install dependencies, and connect to the gated warehouse.
import os
import subprocess
import numpy as np
import pandas as pd

REPO_DIR = "/content/Flyrank"
REPO_URL = "https://github.com/Samarjamal326/Flyrank.git"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

subprocess.run(
    ["pip", "-q", "install", "duckdb", "scikit-learn", "pandas", "numpy"],
    check=True
)

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found. Add your Hugging Face READ token in Colab Secrets "
        "with the name HF_TOKEN and enable notebook access."
    )

import duckdb
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN getvariable('hf_token'))"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

os.makedirs("work/outputs", exist_ok=True)

print("Repository:", os.getcwd())
print("Warehouse connected.")


Repository: /content/Flyrank
Warehouse connected.


In [2]:
# Build the same February decision frame used in ML-08.
feb_sql = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS feb_impressions,
    SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS feb_clicks,
    SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE)
        / NULLIF(SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE), 0)
        AS feb_avg_position
FROM {FEB}
GROUP BY 1, 2
"""

feb = con.execute(feb_sql).df()

feb["feb_ctr"] = np.where(
    feb["feb_impressions"] > 0,
    100.0 * feb["feb_clicks"] / feb["feb_impressions"],
    np.nan,
)

def position_tier(x):
    if x <= 3:
        return "top_3"
    if x <= 10:
        return "page_1"
    if x <= 20:
        return "striking"
    if x <= 50:
        return "page_3_5"
    return "deep"

feb = feb.loc[
    (feb["feb_impressions"].fillna(0) >= 100)
    & feb["feb_avg_position"].notna()
    & np.isfinite(feb["feb_avg_position"])
].copy()

feb["position_tier"] = feb["feb_avg_position"].map(position_tier)

print(f"February decision rows: {len(feb):,}")
display(feb.head())


February decision rows: 80,322


,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_avg_position,feb_ctr,position_tier
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,12.448161,0.000000,striking
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.316508,0.818554,page_1
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,9.966926,0.000000,page_1
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,41.814739,0.102354,page_3_5
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,10.307216,0.206186,striking


In [ ]:
# Build March outcome and join to February.
mar_sql = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS mar_impressions,
    SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS mar_clicks
FROM {MAR}
GROUP BY 1, 2
"""

mar = con.execute(mar_sql).df()

mar["future_ctr"] = np.where(
    mar["mar_impressions"] > 0,
    100.0 * mar["mar_clicks"] / mar["mar_impressions"],
    np.nan,
)

mar = mar.loc[
    mar["mar_impressions"].fillna(0) >= 100,
    ["client_hash_id", "content_hash_id", "mar_impressions", "mar_clicks", "future_ctr"]
].copy()

data = feb.merge(
    mar,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print(f"Matched February → March rows with measurable outcome: {len(data):,}")
display(data.head())


In [ ]:
# Define the honest grouped split first.
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
group_train_idx, group_test_idx = next(
    gss.split(data, groups=data["client_hash_id"])
)

group_train = data.iloc[group_train_idx].copy()
group_test = data.iloc[group_test_idx].copy()

train_clients = set(group_train["client_hash_id"])
test_clients = set(group_test["client_hash_id"])

print(f"Grouped train rows: {len(group_train):,}")
print(f"Grouped test rows:  {len(group_test):,}")
print(f"Train clients: {len(train_clients):,}")
print(f"Test clients:  {len(test_clients):,}")
print("Client overlap:", len(train_clients & test_clients))


In [ ]:
# Freeze the position-tier benchmark using ONLY grouped-training clients.
tier_benchmark = (
    group_train.groupby("position_tier", observed=True)["feb_ctr"]
    .median()
    .rename("tier_median_ctr")
)

data = data.merge(
    tier_benchmark,
    on="position_tier",
    how="left",
    validate="many_to_one"
)

data["y"] = (data["future_ctr"] < data["tier_median_ctr"]).astype(int)

print("Frozen February training-only benchmark:")
display(tier_benchmark.to_frame())

print("\nLabel distribution:")
display(data["y"].value_counts(normalize=True).rename("share").to_frame())

print(
    "\nImportant: March is used only to construct the fixed outcome label; "
    "it is never used as a model feature."
)


### Leakage audit before fitting

The model is restricted to five February fields:

- `feb_impressions`
- `feb_clicks`
- `feb_ctr`
- `feb_avg_position`
- `content_age_days_feb` is not available in the warehouse aggregation used here, so it is **not** included in this audit rerun.

The following are explicitly excluded:

- `future_ctr`
- `mar_impressions`
- `mar_clicks`
- `tier_median_ctr` as a model feature
- `y`
- `trend_direction`
- `trend_pct`
- `client_hash_id`
- `content_hash_id`

The February position-tier benchmark is allowed to define the proxy label, but it is **not passed into the model as a feature**.


In [ ]:
# Add the same decision-time features used by the audited model.
feature_cols = [
    "feb_impressions",
    "feb_clicks",
    "feb_ctr",
    "feb_avg_position",
]

excluded_cols = [
    "future_ctr",
    "mar_impressions",
    "mar_clicks",
    "tier_median_ctr",
    "y",
    "trend_direction",
    "trend_pct",
    "client_hash_id",
    "content_hash_id",
]

print("Model features:", feature_cols)
print("Explicitly excluded leakage / identifier fields:", excluded_cols)

missing = [c for c in feature_cols if c not in data.columns]
assert not missing, f"Missing required feature columns: {missing}"

assert not set(feature_cols) & set(excluded_cols)
assert data[feature_cols].notna().all().all(), "Unexpected missing model features."

leakage_scan = pd.DataFrame({
    "field": feature_cols + excluded_cols,
    "role": (
        ["feature"] * len(feature_cols)
        + ["excluded"] * len(excluded_cols)
    ),
    "contains_future_or_label_signal": [
        any(token in c.lower() for token in ["future", "mar_", "trend", "target", "label", "y"])
        for c in feature_cols + excluded_cols
    ],
})

display(leakage_scan)

assert not leakage_scan.loc[
    leakage_scan["role"] == "feature",
    "contains_future_or_label_signal"
].any()

print("Leakage audit: PASS for the declared feature list.")


### Before: naive random row split

This is intentionally shown as the weaker design. The random row split can place pages from the same client in both training and test sets. A high score here would therefore be less informative about generalization to unseen clients.


In [ ]:
# Naive random row split: BEFORE.
row_train, row_test = train_test_split(
    data,
    test_size=0.20,
    random_state=42,
    stratify=data["y"],
)

row_train_clients = set(row_train["client_hash_id"])
row_test_clients = set(row_test["client_hash_id"])
row_overlap = row_train_clients & row_test_clients

print(f"Naive train rows: {len(row_train):,}")
print(f"Naive test rows:  {len(row_test):,}")
print(f"Clients in both train and test: {len(row_overlap):,}")
print(
    f"Client overlap rate among test clients: "
    f"{len(row_overlap) / max(1, len(row_test_clients)):.1%}"
)


In [ ]:
def make_model():
    return Pipeline([
        ("scale", StandardScaler()),
        ("logreg", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )),
    ])

def precision_at_k(y_true, scores, k):
    k = min(k, len(y_true))
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

def ranking_metrics(y_true, scores):
    return {
        "Precision@10": precision_at_k(y_true, scores, 10),
        "Precision@50": precision_at_k(y_true, scores, 50),
        "Average precision": average_precision_score(y_true, scores),
        "ROC-AUC": roc_auc_score(y_true, scores),
        "Base rate": float(np.mean(y_true)),
    }

def baseline_scores(frame):
    gap = (
        frame["tier_median_ctr"] - frame["feb_ctr"]
    ).clip(lower=0)
    return gap * np.log1p(frame["feb_impressions"])

# Fit/evaluate the naive split.
row_model = make_model()
row_model.fit(row_train[feature_cols], row_train["y"])

row_model_scores = row_model.predict_proba(row_test[feature_cols])[:, 1]
row_baseline_scores = baseline_scores(row_test)

row_model_metrics = ranking_metrics(row_test["y"], row_model_scores)
row_baseline_metrics = ranking_metrics(row_test["y"], row_baseline_scores)

before_after_naive = pd.DataFrame([
    {"validation": "Naive row split — model", **row_model_metrics},
    {"validation": "Naive row split — baseline", **row_baseline_metrics},
])

display(before_after_naive.round(3))


### After: client-grouped split

This is the honest split used by Week 5: the model is evaluated only on clients that were not present during fitting. This better matches a claim about generalization across clients and is the validation design we should prefer for the capstone.


In [ ]:
# Fit/evaluate the client-grouped split: AFTER.
group_model = make_model()
group_model.fit(group_train[feature_cols], group_train["y"])

group_model_scores = group_model.predict_proba(group_test[feature_cols])[:, 1]
group_baseline_scores = baseline_scores(group_test)

group_model_metrics = ranking_metrics(group_test["y"], group_model_scores)
group_baseline_metrics = ranking_metrics(group_test["y"], group_baseline_scores)

comparison = pd.DataFrame([
    {"validation": "Naive row split — model", **row_model_metrics},
    {"validation": "Naive row split — baseline", **row_baseline_metrics},
    {"validation": "Client-grouped — model", **group_model_metrics},
    {"validation": "Client-grouped — baseline", **group_baseline_metrics},
])

display(comparison.round(3))

print("\nBefore → after model Precision@10:")
print(
    f"{row_model_metrics['Precision@10']:.3f} → "
    f"{group_model_metrics['Precision@10']:.3f}"
)


In [ ]:
# Show the validation effect directly.
before = row_model_metrics["Precision@10"]
after = group_model_metrics["Precision@10"]

validation_gap = after - before

print("=== Validation audit ===")
print(f"Naive row-split Precision@10: {before:.3f}")
print(f"Client-grouped Precision@10:   {after:.3f}")
print(f"Absolute change:               {validation_gap:+.3f}")

if after < before:
    print(
        "\nInterpretation: the honest grouped split reduced the measured score. "
        "That is expected when the naive split benefits from client overlap; "
        "the grouped result is the more credible estimate for unseen-client use."
    )
else:
    print(
        "\nInterpretation: the grouped split did not reduce Precision@10. "
        "This is encouraging, but the grouped result remains the preferred "
        "estimate because it matches the intended unseen-client validation design."
    )


## 3. Leakage audit

### Mechanical checks

A validation audit is stronger when the notebook proves the split and feature exclusions rather than only describing them.

The key checks are:

1. **No client overlap** in the honest grouped split.
2. **No future-window columns** in `feature_cols`.
3. **No label-derived columns** in `feature_cols`.
4. The March outcome is used only after the February decision information is fixed.
5. The position-tier benchmark used to define the target is estimated from grouped-training clients only.


In [ ]:
# Final mechanical leakage checks.
assert len(train_clients & test_clients) == 0

for forbidden in [
    "future_ctr",
    "mar_impressions",
    "mar_clicks",
    "tier_median_ctr",
    "y",
    "trend_direction",
    "trend_pct",
]:
    assert forbidden not in feature_cols, f"Forbidden feature leaked: {forbidden}"

assert "client_hash_id" not in feature_cols
assert "content_hash_id" not in feature_cols

print("1. Client overlap check: PASS")
print("2. Future-window feature check: PASS")
print("3. Label-derived feature check: PASS")
print("4. Identifier feature check: PASS")
print("5. Training-only benchmark construction: PASS")


## 4. Real failure examples

The failure review uses the **client-grouped test set**, not the easier random split.

A false positive means the model ranked a page highly as an opportunity, but the March outcome did **not** fall below the benchmark.

A false negative means the model ranked a page lower, but its March outcome **did** fall below the benchmark.

These examples are diagnostic. They do not establish why the model failed.


In [ ]:
# Build an interpretable error table on the honest grouped test set.
error_review = group_test[
    [
        "client_hash_id",
        "content_hash_id",
        "position_tier",
        "feb_impressions",
        "feb_clicks",
        "feb_ctr",
        "feb_avg_position",
        "future_ctr",
        "tier_median_ctr",
        "y",
    ]
].copy()

error_review["model_score"] = group_model_scores
error_review["model_pred"] = (error_review["model_score"] >= 0.5).astype(int)

false_positive = error_review[
    (error_review["model_pred"] == 1) & (error_review["y"] == 0)
].sort_values("model_score", ascending=False)

false_negative = error_review[
    (error_review["model_pred"] == 0) & (error_review["y"] == 1)
].sort_values("model_score", ascending=False)

cols = [
    "client_hash_id",
    "content_hash_id",
    "position_tier",
    "feb_impressions",
    "feb_ctr",
    "future_ctr",
    "tier_median_ctr",
    "model_score",
    "y",
]

print("Top false positives:")
display(false_positive[cols].head(5).round(4))

print("Top false negatives:")
display(false_negative[cols].head(5).round(4))


In [ ]:
# Check whether low-volume pages are overrepresented in errors.
error_review["volume_bucket"] = pd.cut(
    error_review["feb_impressions"],
    bins=[99, 249, 499, 999, 4999, np.inf],
    labels=["100-249", "250-499", "500-999", "1k-4.9k", "5k+"],
)

error_review["error_type"] = np.select(
    [
        (error_review["model_pred"] == 1) & (error_review["y"] == 0),
        (error_review["model_pred"] == 0) & (error_review["y"] == 1),
    ],
    ["false_positive", "false_negative"],
    default="correct",
)

error_by_volume = (
    error_review.groupby("volume_bucket", observed=True)
    .agg(
        n=("y", "size"),
        errors=("error_type", lambda s: (s != "correct").sum()),
        false_positives=("error_type", lambda s: (s == "false_positive").sum()),
        false_negatives=("error_type", lambda s: (s == "false_negative").sum()),
    )
)

error_by_volume["error_rate"] = error_by_volume["errors"] / error_by_volume["n"]

display(error_by_volume.round(3))

print(
    "Diagnostic note: low-volume CTR is intrinsically noisier, so error concentration "
    "there would be a limitation of the proxy outcome rather than evidence of a causal failure."
)


## Claim rewrite

### Week-5 claim that needs tightening

A claim such as:

> “The Logistic Regression model improves CTR opportunity identification and can identify pages that should be refreshed.”

goes beyond what this experiment proves.

### Audited, evidence-matched version

> **On the measured March outcome and the client-held-out test set, Logistic Regression provided a ranked score for pages whose future CTR fell below the February position-tier benchmark. Its Precision@10 was measured against the frozen Week-4 baseline on the same held-out clients. The result supports decision-support prioritization of review candidates; it does not establish that refreshing a page will cause CTR to improve.**

### Validation claim

Instead of:

> “The model generalizes well.”

use:

> **“The model was evaluated on clients not used for fitting, providing a more conservative estimate of unseen-client ranking performance than a random row split.”**

### Research-paper interpretation

The paper's Growth Prediction result should likewise be read as a measured predictive result under its stated same-brand and unseen-brand tests, not as proof that the listed predictors cause growth. The paper itself distinguishes pattern evidence from causal proof. citeturn5view2turn4view0


In [ ]:
# Automatically print a compact claim audit summary.
print("=== ML-09 CLAIM AUDIT SUMMARY ===")
print(
    f"Naive row-split model Precision@10: {row_model_metrics['Precision@10']:.3f}"
)
print(
    f"Client-grouped model Precision@10:   {group_model_metrics['Precision@10']:.3f}"
)
print(
    f"Grouped baseline Precision@10:        {group_baseline_metrics['Precision@10']:.3f}"
)
print(f"Client overlap in honest split:       {len(train_clients & test_clients)}")

print("\nSafe conclusion:")
if group_model_metrics["Precision@10"] > group_baseline_metrics["Precision@10"]:
    print(
        "On the held-out clients, the model measured higher Precision@10 than the "
        "baseline under this proxy outcome."
    )
else:
    print(
        "On the held-out clients, the model did not measure higher Precision@10 "
        "than the baseline under this proxy outcome."
    )

print(
    "This is an observed predictive-ranking result for decision support, "
    "not evidence of a causal refresh effect."
)


## 5. Self-check

Before committing:

- [x] Two research-paper findings are named with constructive methodology questions.
- [x] The Week-5 model is re-run.
- [x] Naive row split is shown as the **before** design.
- [x] Client-grouped split is shown as the **after / honest** design.
- [x] Both designs use the same frozen target benchmark.
- [x] Leakage checks are mechanical, not just narrative.
- [x] Future-window and label-derived fields are excluded from model features.
- [x] Real false-positive and false-negative examples are reviewed.
- [x] Claims are rewritten as observed / measured / directional / decision-support.
- [x] No causal claim is made about refreshing pages.
- [x] The final comparison uses the same Precision@K definitions.

### Deliverable

Save this executed notebook as:

`work/notebooks/w06_validation_audit.ipynb`

Then commit it to the `main` branch of the Flyrank repository and submit the repository URL.
